# 📚 Smart Research Assistant using RAG + LLM

This notebook builds a Retrieval-Augmented Generation (RAG) system step by step.

**Run cells from top to bottom in Google Colab.**

## 🔹 Cell 1: Install Required Libraries
We install LangChain, vector database (FAISS), embedding models, and PDF loaders.

In [ ]:

!pip install -q langchain langchain-community langchain-openai
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q langchain-google-genai


## 🔹 Cell 2: Upload PDF Documents
Upload the documents that the assistant will learn from.

In [ ]:

from google.colab import files
uploaded = files.upload()
uploaded.keys()


## 🔹 Cell 3: Load PDF Files
Each PDF page is loaded as a document object.

In [ ]:

from langchain.document_loaders import PyPDFLoader

documents = []
for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    documents.extend(loader.load())

print("Total pages loaded:", len(documents))


## 🔹 Cell 4: Split Documents into Chunks
Chunking improves semantic retrieval and avoids context loss.

In [ ]:

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print("Total chunks created:", len(chunks))


## 🔹 Cell 5: Create Embeddings
Embeddings convert text into numerical vectors for semantic search.

In [ ]:

from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


## 🔹 Cell 6: Build Vector Database (FAISS)
FAISS stores embeddings and allows fast similarity search.

In [ ]:

from langchain.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embeddings)
print("Vector database created")


## 🔹 Cell 7: Load LLM (Google Gemini)
Gemini is used to generate answers from retrieved context.

In [ ]:

import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "PASTE_YOUR_API_KEY"

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=0
)


## 🔹 Cell 8: Create RAG Pipeline
This connects the retriever with the LLM.

In [ ]:

from langchain.chains import RetrievalQA

retriever = vector_db.as_retriever(search_kwargs={"k": 4})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG pipeline ready")


## 🔹 Cell 9: Ask a Question with Citations
The answer is generated using retrieved document chunks.

In [ ]:

query = "Summarize the main topic of the document"

response = qa_chain(query)

print("ANSWER:\n", response["result"])
print("\nSOURCES:")
for doc in response["source_documents"]:
    print(f"Page {doc.metadata.get('page')}")


## 🔹 Cell 10: Interactive Chat Mode
Ask unlimited questions until you type exit.

In [ ]:

while True:
    q = input("Ask a question (type exit to stop): ")
    if q.lower() == "exit":
        break
    res = qa_chain(q)
    print("\nAnswer:", res["result"])
